## 1. GPU check

Neu chay tren Colab, hay bat `Runtime > Change runtime type > T4 GPU` truoc.


In [ ]:
import os
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)

if '/content' in os.getcwd() and not gpus:
    raise RuntimeError(
        'No GPU detected in Colab. Go to Runtime > Change runtime type > T4 GPU, then rerun from the top.'
    )

if gpus:
    try:
        print('GPU name:', tf.config.experimental.get_device_details(gpus[0]).get('device_name', '<unknown>'))
    except Exception as exc:
        print('Could not read GPU name:', exc)


## 2. Repo bootstrap

Cell nay se clone repo neu dang chay tren Colab. Neu chay local trong repo thi no se tu tim project root hien tai.


In [ ]:
# import os
# import subprocess
# import sys
# from pathlib import Path

# REPO_URL = 'https://github.com/HoangHumg1210/hoankiem-air-quality-.git'
# REPO_NAME = 'hoankiem-air-quality-forecasting'

# cwd = Path.cwd().resolve()
# if str(cwd).startswith('/content'):
#     project_root = Path('/content') / REPO_NAME
#     os.chdir('/content')
#     if not project_root.exists():
#         subprocess.run(['git', 'clone', REPO_URL, REPO_NAME], check=True)
# else:
#     project_root = cwd if (cwd / 'src').exists() else cwd.parent

# os.chdir(project_root)
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

# required_paths = [
#     project_root / 'src' / 'data_utils.py',
#     project_root / 'data' / 'processed' / 'data2225_done.csv',
# ]
# missing_paths = [str(path) for path in required_paths if not path.exists()]
# if missing_paths:
#     raise FileNotFoundError('Missing required project files: ' + ', '.join(missing_paths))

# print('PROJECT_ROOT =', project_root)
# print('sys.path[0] =', sys.path[0])
# %cd /content

# !rm -rf hoankiem-air-quality-forecasting

# !git clone https://github.com/HoangHumg1210/hoankiem-air-quality-forecasting.git

# %cd hoankiem-air-quality-forecasting

# !ls

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("cwd =", os.getcwd())

## 3. Imports

Neu import package nao fail, ban moi can cai them package do.


In [ ]:
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, GRU, Input
from tensorflow.keras.metrics import MeanAbsoluteError
from tensorflow.keras.optimizers import Adam

from src.data_utils import (
    CFG,
    evaluate_by_horizon,
    evaluate_regression,
    inverse_y,
    prepare_dataset,
    prepare_walk_forward_datasets,
    set_seed,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
print('Imports OK')


## 4. Configs

Sua cell nay de doi hyperparameters cho smoke run hoac full training.


In [ ]:
@dataclass
class GRUExperimentConfig:
    gru_units: tuple[int, ...] = (192, 96)
    dense_units: Optional[int] = 64
    dropout: float = 0.1
    learning_rate: float = 1e-3
    batch_size: int = 64
    epochs: int = 40
    patience: int = 10
    min_lr: float = 1e-5
    step_to_plot: int = 1
    confidence_z: float = 1.96
    horizon_weights: Optional[tuple[float, ...]] = None
    peak_quantile: float = 0.90
    lambda_peak: float = 2.0


default_horizon_weights = tuple([3.0] * 24 + [1.75] * 24 + [1.0] * 24)
data_cfg = replace(CFG, lookback=168, horizon=72, target_transform='sqrt')
walk_forward_folds = 2

smoke_cfg = GRUExperimentConfig(
    gru_units=(96,),
    dense_units=32,
    dropout=0.05,
    batch_size=128,
    epochs=2,
    patience=2,
    horizon_weights=default_horizon_weights,
)

full_cfg = GRUExperimentConfig(
    gru_units=(192, 96),
    dense_units=64,
    dropout=0.1,
    batch_size=64,
    epochs=40,
    patience=10,
    horizon_weights=default_horizon_weights,
    peak_quantile=0.90,
    lambda_peak=2.0,
)

if len(default_horizon_weights) != data_cfg.horizon:
    raise ValueError('default_horizon_weights must match horizon=72.')

print(data_cfg)
print('walk_forward_folds =', walk_forward_folds)
print('Smoke config:', smoke_cfg)
print('Full config :', full_cfg)


## 5. Prepare data

Cell nay chay pipeline `prepare_dataset()` va cho ban thay ngay shape train / val / test.


In [ ]:
set_seed(data_cfg.seed)
artifacts = prepare_dataset(data_cfg)

print('Train seq:', artifacts['X_train_seq'].shape, artifacts['y_train_seq'].shape)
print('Val seq  :', artifacts['X_val_seq'].shape, artifacts['y_val_seq'].shape)
print('Test seq :', artifacts['X_test_seq'].shape, artifacts['y_test_seq'].shape)
print('n_features:', artifacts['n_features'])


## 6. Build GRU model

Neu muon doi architecture, ban sua cell function nay.


In [ ]:
def resolve_horizon_weights(cfg: GRUExperimentConfig, horizon: int) -> np.ndarray:
    if cfg.horizon_weights is None:
        return np.ones(horizon, dtype=np.float32)

    weights = np.asarray(cfg.horizon_weights, dtype=np.float32)
    if weights.shape[0] != horizon:
        raise ValueError(f'horizon_weights length {weights.shape[0]} must equal horizon {horizon}.')
    return weights


def compute_peak_thresholds(artifacts: dict, peak_quantile: float) -> tuple[float, float]:
    y_train_raw = inverse_y(artifacts['y_train_seq'], artifacts['target_transformer'])
    peak_threshold = float(np.quantile(y_train_raw.reshape(-1), peak_quantile))
    peak_threshold_scaled = float(
        artifacts['target_transformer'].transform(np.array([[peak_threshold]], dtype=np.float32))[0, 0]
    )
    return peak_threshold, peak_threshold_scaled


def build_weighted_peak_loss(horizon_weights, peak_threshold_scaled: float, lambda_peak: float):
    weights = tf.constant(np.asarray(horizon_weights, dtype=np.float32).reshape(1, -1), dtype=tf.float32)
    peak_threshold_scaled = tf.constant(float(peak_threshold_scaled), dtype=tf.float32)
    lambda_peak = tf.constant(float(lambda_peak), dtype=tf.float32)

    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        squared_error = tf.square(y_true - y_pred)
        weighted_horizon_mse = tf.reduce_mean(squared_error * weights)

        peak_mask = tf.cast(y_true >= peak_threshold_scaled, tf.float32)
        underprediction = tf.nn.relu(y_true - y_pred)
        peak_underprediction_penalty = tf.reduce_mean(tf.square(underprediction) * peak_mask)

        return weighted_horizon_mse + lambda_peak * peak_underprediction_penalty

    return loss


def build_gru_model(
    input_shape: tuple[int, int],
    horizon: int,
    cfg: GRUExperimentConfig,
    horizon_weights,
    peak_threshold_scaled: float,
) -> Model:
    model = Sequential(name='gru_forecaster')
    model.add(Input(shape=input_shape))

    for idx, units in enumerate(cfg.gru_units):
        return_sequences = idx < len(cfg.gru_units) - 1
        model.add(GRU(units, return_sequences=return_sequences))
        if cfg.dropout > 0:
            model.add(Dropout(cfg.dropout))

    if cfg.dense_units is not None:
        model.add(Dense(cfg.dense_units, activation='relu'))
        if cfg.dropout > 0:
            model.add(Dropout(cfg.dropout))

    model.add(Dense(horizon, name='forecast'))
    model.compile(
        optimizer=Adam(learning_rate=cfg.learning_rate),
        loss=build_weighted_peak_loss(horizon_weights, peak_threshold_scaled, cfg.lambda_peak),
        metrics=[MeanAbsoluteError(name='mae')],
    )
    return model


_preview_weights = resolve_horizon_weights(full_cfg, artifacts['cfg'].horizon)
_preview_peak_threshold, _preview_peak_threshold_scaled = compute_peak_thresholds(artifacts, full_cfg.peak_quantile)
print('Preview peak threshold:', round(_preview_peak_threshold, 3))

model_preview = build_gru_model(
    input_shape=(artifacts['cfg'].lookback, artifacts['n_features']),
    horizon=artifacts['cfg'].horizon,
    cfg=full_cfg,
    horizon_weights=_preview_weights,
    peak_threshold_scaled=_preview_peak_threshold_scaled,
)
model_preview.summary()


## 7. Train helpers

Cell nay tach rieng logic train de ban de thay callback va data train / val.


In [ ]:
def train_gru_model(artifacts: dict, cfg: GRUExperimentConfig):
    horizon_weights = resolve_horizon_weights(cfg, artifacts['cfg'].horizon)
    peak_threshold, peak_threshold_scaled = compute_peak_thresholds(artifacts, cfg.peak_quantile)

    model = build_gru_model(
        input_shape=(artifacts['cfg'].lookback, artifacts['n_features']),
        horizon=artifacts['cfg'].horizon,
        cfg=cfg,
        horizon_weights=horizon_weights,
        peak_threshold_scaled=peak_threshold_scaled,
    )

    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=cfg.patience,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=max(2, cfg.patience // 2),
            min_lr=cfg.min_lr,
            verbose=1,
        ),
    ]

    history = model.fit(
        artifacts['X_train_seq'],
        artifacts['y_train_seq'],
        validation_data=(artifacts['X_val_seq'], artifacts['y_val_seq']),
        epochs=cfg.epochs,
        batch_size=cfg.batch_size,
        callbacks=callbacks,
        verbose=1,
    )

    train_info = {
        'horizon_weights': horizon_weights,
        'peak_threshold': peak_threshold,
        'peak_threshold_scaled': peak_threshold_scaled,
        'lambda_peak': cfg.lambda_peak,
    }
    return model, history, train_info


## 8. Evaluate helpers

Cell nay tach rieng phan predict, metric va tao bang forecast.


In [ ]:
def predict_split(model: Model, artifacts: dict, split: str = 'test'):
    X_seq = artifacts[f'X_{split}_seq']
    y_seq = artifacts[f'y_{split}_seq']
    times = pd.DatetimeIndex(artifacts[f'{split}_times'])

    y_pred_scaled = model.predict(X_seq, verbose=0)
    y_pred = inverse_y(y_pred_scaled, artifacts['target_transformer'])
    y_true = inverse_y(y_seq, artifacts['target_transformer'])
    return y_true, y_pred, times


def build_forecast_frame(y_true, y_pred, times, step=1, residual_std=None, confidence_z=1.96):
    step_idx = step - 1
    frame = pd.DataFrame({
        'time': pd.DatetimeIndex(times),
        'actual': y_true[:, step_idx],
        'predicted': y_pred[:, step_idx],
    })
    if residual_std is not None:
        half_width = confidence_z * residual_std
        frame['lower'] = np.clip(frame['predicted'] - half_width, 0, None)
        frame['upper'] = frame['predicted'] + half_width
    return frame


def build_daily_metrics_frame(forecast_frame: pd.DataFrame, split: str):
    daily_frame = forecast_frame.copy()
    daily_frame['time'] = pd.to_datetime(daily_frame['time'])
    daily_frame = daily_frame.set_index('time').resample('1D').mean(numeric_only=True).dropna().reset_index()
    daily_metrics = evaluate_regression(
        daily_frame['actual'].values,
        daily_frame['predicted'].values,
        name=f'{split.upper()}(daily mean)',
    )
    daily_metrics_frame = pd.DataFrame([
        {'split': split, 'scope': 'daily_mean', **daily_metrics},
    ])
    return daily_frame, daily_metrics, daily_metrics_frame


def evaluate_predictions(model: Model, artifacts: dict, split: str = 'test', plot_step: int = 1, confidence_z: float = 1.96):
    y_true, y_pred, times = predict_split(model, artifacts, split=split)
    metrics_all = evaluate_regression(y_true, y_pred, name=f'{split.upper()}(all horizons)')
    metrics_step = evaluate_regression(y_true[:, 0], y_pred[:, 0], name=f'{split.upper()}(step 1)')
    by_horizon = evaluate_by_horizon(y_true, y_pred)

    residual_std = None
    if split == 'test':
        y_true_val, y_pred_val, _ = predict_split(model, artifacts, split='val')
        residual_std = float(np.std(y_true_val[:, plot_step - 1] - y_pred_val[:, plot_step - 1]))

    forecast_frame = build_forecast_frame(
        y_true=y_true,
        y_pred=y_pred,
        times=times,
        step=plot_step,
        residual_std=residual_std,
        confidence_z=confidence_z,
    )
    daily_frame, daily_metrics, daily_metrics_frame = build_daily_metrics_frame(forecast_frame, split=split)

    metrics_frame = pd.concat([
        pd.DataFrame([
            {'split': split, 'scope': 'all_horizons', **metrics_all},
            {'split': split, 'scope': 'step_1', **metrics_step},
        ]),
        daily_metrics_frame,
    ], ignore_index=True)

    return {
        'y_true': y_true,
        'y_pred': y_pred,
        'times': times,
        'metrics_all': metrics_all,
        'metrics_step': metrics_step,
        'daily_metrics': daily_metrics,
        'metrics_frame': metrics_frame,
        'by_horizon': by_horizon,
        'forecast_frame': forecast_frame,
        'daily_frame': daily_frame,
    }


def run_walk_forward_backtest(data_cfg, cfg: GRUExperimentConfig, max_folds: int = 2):
    folds = prepare_walk_forward_datasets(data_cfg, max_folds=max_folds)
    fold_runs = []
    metric_frames = []

    for fold in folds:
        fold_model, fold_history, fold_train_info = train_gru_model(fold, cfg)
        fold_val = evaluate_predictions(
            fold_model,
            fold,
            split='val',
            plot_step=cfg.step_to_plot,
            confidence_z=cfg.confidence_z,
        )

        fold_metrics = fold_val['metrics_frame'].copy()
        fold_metrics.insert(0, 'evaluation', 'walk_forward')
        fold_metrics.insert(1, 'fold', fold['fold'])
        metric_frames.append(fold_metrics)
        fold_runs.append({
            'fold': fold['fold'],
            'artifacts': fold,
            'model': fold_model,
            'history': fold_history,
            'train_info': fold_train_info,
            'val_results': fold_val,
        })

    walk_forward_metrics_df = pd.concat(metric_frames, ignore_index=True)
    walk_forward_summary_df = (
        walk_forward_metrics_df.groupby('scope', as_index=False)[['mae', 'mse', 'rmse']].mean()
        .sort_values('scope')
        .reset_index(drop=True)
    )
    return fold_runs, walk_forward_metrics_df, walk_forward_summary_df


## 9. Plot helpers

Cell nay gom toan bo phan visualize de ban de doi style plot.


In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    history_df = pd.DataFrame(history.history)

    history_df[['loss', 'val_loss']].plot(ax=axes[0], title='Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    history_df[['mae', 'val_mae']].plot(ax=axes[1], title='MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    return fig


def plot_forecast(forecast_frame: pd.DataFrame, title: str = 'GRU forecast on test', recent_days: int = 14):
    plot_df = forecast_frame.copy().sort_values('time')
    plot_df['time'] = pd.to_datetime(plot_df['time'])
    plot_df = plot_df.set_index('time')

    daily_df = plot_df.resample('1D').mean(numeric_only=True)
    recent_start = plot_df.index.max() - pd.Timedelta(days=recent_days)
    recent_df = plot_df.loc[plot_df.index >= recent_start].copy()
    smooth_window = min(24, max(3, len(recent_df) // 20))
    recent_df['actual_smooth'] = recent_df['actual'].rolling(smooth_window, min_periods=1).mean()
    recent_df['predicted_smooth'] = recent_df['predicted'].rolling(smooth_window, min_periods=1).mean()

    fig, axes = plt.subplots(2, 1, figsize=(15, 9), gridspec_kw={'height_ratios': [1.15, 1.0]})

    axes[0].plot(daily_df.index, daily_df['actual'], label='Actual daily mean', linewidth=2.5, color='#1f77b4')
    axes[0].plot(daily_df.index, daily_df['predicted'], label='GRU daily mean', linewidth=2.5, color='#ff7f0e')
    if {'lower', 'upper'}.issubset(daily_df.columns):
        axes[0].fill_between(daily_df.index, daily_df['lower'], daily_df['upper'], color='#ff7f0e', alpha=0.12, label='Approx. interval')
    axes[0].set_title(f'{title} - full test (daily mean)')
    axes[0].set_ylabel('PM2.5')
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(loc='upper left', ncol=3, frameon=True)

    axes[1].plot(recent_df.index, recent_df['actual'], linewidth=0.9, color='#1f77b4', alpha=0.25, label='Actual hourly')
    axes[1].plot(recent_df.index, recent_df['predicted'], linewidth=1.0, color='#ff7f0e', alpha=0.35, label='GRU hourly')
    axes[1].plot(recent_df.index, recent_df['actual_smooth'], linewidth=2.5, color='#1f77b4', label=f'Actual smooth ({smooth_window}h)')
    axes[1].plot(recent_df.index, recent_df['predicted_smooth'], linewidth=2.5, color='#ff7f0e', label=f'GRU smooth ({smooth_window}h)')
    if {'lower', 'upper'}.issubset(recent_df.columns):
        axes[1].fill_between(recent_df.index, recent_df['lower'], recent_df['upper'], color='#ff7f0e', alpha=0.10, label='Approx. interval')
    axes[1].set_title(f'Recent hourly zoom ({recent_days} days)')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('PM2.5')
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(loc='upper left', ncol=4, frameon=True)

    fig.tight_layout()
    return fig


## 10. Smoke run

Chay cell nay truoc de bat loi moi truong / path / GPU.


In [ ]:
smoke_model, smoke_history, smoke_train_info = train_gru_model(artifacts, smoke_cfg)
print('Smoke peak threshold:', round(smoke_train_info['peak_threshold'], 3))
smoke_val = evaluate_predictions(smoke_model, artifacts, split='val', plot_step=smoke_cfg.step_to_plot, confidence_z=smoke_cfg.confidence_z)
smoke_test = evaluate_predictions(smoke_model, artifacts, split='test', plot_step=smoke_cfg.step_to_plot, confidence_z=smoke_cfg.confidence_z)
smoke_metrics = pd.concat([smoke_val['metrics_frame'], smoke_test['metrics_frame']], ignore_index=True)
smoke_metrics


## 11. Full training

Sau khi smoke run OK, chay full training.


In [ ]:
model, history, train_info = train_gru_model(artifacts, full_cfg)
print('Full peak threshold:', round(train_info['peak_threshold'], 3))
val_results = evaluate_predictions(model, artifacts, split='val', plot_step=full_cfg.step_to_plot, confidence_z=full_cfg.confidence_z)
test_results = evaluate_predictions(model, artifacts, split='test', plot_step=full_cfg.step_to_plot, confidence_z=full_cfg.confidence_z)


In [ ]:
metrics_df = pd.concat([val_results['metrics_frame'], test_results['metrics_frame']], ignore_index=True)
metrics_df.insert(0, 'evaluation', 'fixed_split')
metrics_df


In [ ]:
test_results['by_horizon'].head(24)


## 12. Walk-forward

Cell nay chay backtest tren 2 folds mac dinh va tong hop metric hourly / daily mean.


In [ ]:
wf_runs, wf_metrics_df, wf_summary_df = run_walk_forward_backtest(
    data_cfg,
    full_cfg,
    max_folds=walk_forward_folds,
)
wf_summary_df


In [ ]:
wf_metrics_df


## 13. Visualize


In [ ]:
plot_training_history(history)
plt.show()


In [ ]:
plot_forecast(test_results['forecast_frame'], title='GRU forecast on test')
plt.show()


## 14. Export results

Cell nay de luu metric va forecast ra CSV neu can.


In [ ]:
output_dir = Path('models') / 'gru_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(output_dir / 'gru_metrics.csv', index=False)
test_results['by_horizon'].to_csv(output_dir / 'gru_test_by_horizon.csv', index=False)
test_results['forecast_frame'].to_csv(output_dir / 'gru_test_forecast.csv', index=False)
test_results['daily_frame'].to_csv(output_dir / 'gru_test_daily_mean.csv', index=False)
wf_metrics_df.to_csv(output_dir / 'gru_walk_forward_metrics.csv', index=False)
wf_summary_df.to_csv(output_dir / 'gru_walk_forward_summary.csv', index=False)

print('Saved to:', output_dir.resolve())
